# Real Estate Price Prediction — Model Training
**Author:** Gargi Mishra | IILM University, Greater Noida

This notebook walks through the full pipeline:
1. Load and explore the dataset
2. Preprocess features
3. Train a Random Forest model
4. Evaluate performance
5. Run a sample prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')

from preprocess import load_data, preprocess

df = load_data('../data/sample_data.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Explore the Data

In [ ]:
df.describe()

In [ ]:
# Price distribution by city
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.groupby('city')['price_lakhs'].median().sort_values().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Median Price by City (₹ Lakhs)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

df['price_lakhs'].hist(bins=30, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Price Distribution')
axes[1].set_xlabel('Price (₹ Lakhs)')

plt.tight_layout()
plt.show()

## 2. Preprocess & Train

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, r2_score

X, y, encoders = preprocess(df)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
r2 = r2_score(y_test, y_pred)

print(f'MAPE  : {mape:.2f}%')
print(f'R²    : {r2:.4f}')

## 3. Feature Importance

In [ ]:
feature_names = X.columns.tolist()
importances = model.feature_importances_

fi = pd.Series(importances, index=feature_names).sort_values(ascending=True)
fi.plot(kind='barh', figsize=(8, 4), color='steelblue', edgecolor='white')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

## 4. Sample Prediction

In [ ]:
from predict import predict_price

sample = {
    'city': 'Bangalore',
    'locality_tier': 'Premium',
    'bhk': 2,
    'area_sqft': 1100,
    'metro_dist_km': 1.5,
    'it_park_dist_km': 3.0,
    'rera_registered': 1,
    'ready_to_move': 1,
    'age_years': 2
}

price = predict_price(sample)
print(f'Predicted Price: ₹{price} Lakhs')